# 3D Toric-Code NQS — hyperparameter sweep on Colab

Runs the **same** `Three_TC.train` pipeline as the NERSC GPU job array. On Colab
you run **each config as its own cell** — one fresh `!python` process per cell,
which is the pattern that runs reliably here (the automated loop could skip runs).

**Run cells 1–5 top to bottom once** (setup), then use the **run cells** at the
bottom: copy one, change `NAME` and the training hyperparameters, and run it.

The network is the **code default** in `Three_TC/model/networks.py` — non-invariant
`[4, 4]`, invariant `[4, 4, 1]` (~2.6k params). Each run does dense-QGT Stochastic
Reconfiguration and prints, every step, `E ± ΔE`, the spread `√Var`, and the figure
of merit **`delta = |E − E_exact| / |E_exact|`** (also logged to W&B).

> First: **Runtime → Change runtime type → GPU**, then run cell 1 to confirm.

**Boundary conditions.** Set `BC = "PBC"` or `"OBC"` in the configure cell. PBC uses the hardcoded `--hz_preset` ED reference; OBC (L=2, N=12) diagonalizes the exact ground state **on the fly** and passes it as `--exact_E0`, so `delta` works for both. OBC runs land in their own `outputs/colab-obc-…` folder.

In [1]:
# 1. Confirm a GPU is attached (Runtime -> Change runtime type -> GPU).
!nvidia-smi -L || echo "NO GPU -- switch the runtime to GPU before continuing."

GPU 0: NVIDIA L4 (UUID: GPU-5b51bb2b-13a2-bb52-fd71-4ed14e1f8d43)


In [2]:
# 2. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

Cloning into '/content/ThreeD_TC'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 409 (delta 186), reused 374 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 1.06 MiB | 1.08 MiB/s, done.
Resolving deltas: 100% (186/186), done.
Already up to date.
cwd: /content/ThreeD_TC


In [3]:
# 3. Install the NQS stack, pinned to match the NERSC env (parity of results).
#    Colab may print dependency-conflict warnings for unrelated preinstalled
#    packages (tensorflow, etc.) -- those are harmless for this pipeline.
!pip install -q jax==0.5.2 jaxlib==0.5.1 \
    jax-cuda12-plugin==0.5.1 jax-cuda12-pjrt==0.5.1 \
    netket==3.16.1.post1 flax==0.10.4 optax numba wandb tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.1/695.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.8/451.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.3/354.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.3/529.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00


In [4]:
# 4. GATE: make sure JAX actually sees the GPU (expect platform 'gpu').
#    If this fails: re-run cell 3, then Runtime -> Restart session, then re-run
#    from cell 2. (x64 is already enabled inside Three_TC/train.py.)
import os
# Allocate GPU memory on demand instead of grabbing ~75% up front -- keeps
# back-to-back runs from fighting over VRAM. Set BEFORE importing jax.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
import jax
print("devices:", jax.devices(), "| backend:", jax.default_backend())
assert jax.default_backend() == "gpu", "JAX is not on the GPU -- see the note above."

devices: [CudaDevice(id=0)] | backend: gpu


## Weights & Biases

`WANDB = True` logs every step's `delta` / `energy` / `energy_abs_err` to a shared
group so all configs plot side by side (a parallel-coordinates plot over `delta` is
the intended view). Use a group name **distinct from your NERSC group** so the two
don't mix — same project, so they're still visible together.

Set `WANDB = False` to skip it entirely and just watch the per-step stdout.

In [5]:
# 5. W&B auth. Set WANDB = False to skip and rely on stdout only.
# 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

WANDB = True
if WANDB:
    import wandb
    wandb.login()   # paste your API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


...

wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sbissena (models-california-institute-of-technology-caltech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Configure — fixed args + a training menu  ← edit here

Set the shared args, the architecture, and the per-run output folder once; every run
cell below picks them up. `ARCH_FLAGS = ""` uses the **code default** net
(non-invariant `[4,4]`, invariant `[4,4,1]`, ~2.6k params) — set it to override. The
printed menu lists **training**-hyperparameter combos to copy into a run cell.

Pick `BC` (`PBC`/`OBC`) here as well: OBC computes its exact `E0` on the fly (cheap at L=2) and writes to a separate `colab-obc-…` folder, while PBC is unchanged.

In [6]:
# === EDIT: fixed args shared by every run ==================================
BC        = "OBC"      # boundary conditions: "PBC" | "OBC"
L         = 2
HX        = 0.2        # transverse field (keep 0.2 for PBC: the presets are at hx=0.2)
HZ        = 0.2        # OBC only: the hz value (PBC takes hz from HZ_PRESET below)
HZ_PRESET = "hard"     # PBC ED reference point: hard | mid | easy  (ignored for OBC)
N_ITER    = 200
N_SAMPLES = 16384
N_CHAINS  = 512
N_SWEEPS  = 36         # Metropolis sweeps between samples (code default 2N = 48 at L=2)

# Architecture: ARCH_FLAGS="" uses the CODE DEFAULT (Three_TC/model/networks.py):
#   non-invariant [4, 4]    (noninv_channels=4, n_noninv=2)
#   invariant     [4, 4, 1] (inv_hidden=(4,4) + final width-1)   -> ~2.6k params
# To try a different net, e.g.:
#   ARCH_FLAGS = "--noninv_channels 8 --n_noninv 1 --inv_hidden 8"
ARCH_FLAGS = ""
ARCH_TAG   = "cnn-4-4_4-4-1"          # label -> output folder + wandb group
# ===========================================================================

# Field / exact-E0 spec. PBC uses the hardcoded --hz_preset reference (hx=0.2).
# OBC has no preset, so we diagonalize the L=2 OBC Hamiltonian ON THE FLY (N=12,
# 2^12 states -> a fraction of a second) with the same verified ED as the repo,
# and pass E0 to --exact_E0 so the `delta` figure of merit works for OBC too.
if BC == "OBC":
    import numpy as np
    from scipy.sparse.linalg import eigsh
    from Three_TC.model.geometry import ThreeD_ToricCodeGeometry
    from model.exact_diag import hamiltonian_linop
    _geo = ThreeD_ToricCodeGeometry(L, L, L, bc="OBC")
    _H, _ = hamiltonian_linop(_geo, hx=HX, hz=HZ, J=1.0)
    E0_EXACT = float(eigsh(_H, k=1, which="SA", return_eigenvectors=False)[0])
    FIELD_FLAGS = f"--hx {HX} --hz {HZ} --exact_E0 {E0_EXACT}"
    POINT_TAG   = f"hx{HX}_hz{HZ}"
    print(f"OBC L={L} on-the-fly ED:  E0_exact = {E0_EXACT:.6f}  (N={_geo.N})")
else:
    FIELD_FLAGS = f"--hx {HX} --hz_preset {HZ_PRESET}"
    POINT_TAG   = HZ_PRESET

WANDB_GROUP = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR     = f"outputs/{WANDB_GROUP}"   # fresh per-(bc, point, arch) folder

# Reference menu of TRAINING hyperparameters to try (copy a line into a run cell):
DT_LRMIN   = [(0.02, 0.002), (0.01, 0.001)]   # (initial lr, final lr)
DIAG_SHIFT = [1e-4, 1e-3, 1e-2]               # SR / QGT regularization

print(f"bc: {BC}   field flags: {FIELD_FLAGS}")
print("arch:", "code default [4,4]/[4,4,1]" if not ARCH_FLAGS else ARCH_FLAGS)
print(f"output folder: {OUT_DIR}/   |   "
      + (f"wandb group: {WANDB_GROUP}" if WANDB else "wandb OFF"))
print("training-hyperparameter menu (copy a line into a run cell):")
for (dt, lrm) in DT_LRMIN:
    for ds in DIAG_SHIFT:
        print(f"  --dt {dt} --lr_min {lrm} --diag_shift {ds}")

OBC L=2 on-the-fly ED:  E0_exact = -14.279396  (N=12)
bc: OBC   field flags: --hx 0.2 --hz 0.2 --exact_E0 -14.27939596796281
arch: code default [4,4]/[4,4,1]
output folder: outputs/colab-obc-hx0.2_hz0.2/   |   wandb group: colab-obc-hx0.2_hz0.2
training-hyperparameter menu (copy a line into a run cell):
  --dt 0.02 --lr_min 0.002 --diag_shift 0.0001
  --dt 0.02 --lr_min 0.002 --diag_shift 0.001
  --dt 0.02 --lr_min 0.002 --diag_shift 0.01
  --dt 0.01 --lr_min 0.001 --diag_shift 0.0001
  --dt 0.01 --lr_min 0.001 --diag_shift 0.001
  --dt 0.01 --lr_min 0.001 --diag_shift 0.01


## Sanity check — one short run

A quick low-cost run to confirm training works end-to-end with the default net. The
`[train]` line should report `n_params=2571` (the `[4,4]/[4,4,1]` net), and you
should see the per-step `delta = ...` stream.

In [ ]:
# 6. One short smoke run (uses ARCH_FLAGS = the default [4,4]/[4,4,1] net).
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter 50 --n_samples 4096 --n_chains 256 \
  --n_sweeps {N_SWEEPS} --qgt dense --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name cfg_smoke {wb}

## Run configs — one cell per run

**This is the reliable path:** each cell is a single fresh `!python` process (no
loop). To run a config: **copy a cell below, change `NAME`, edit the training flags
(`--dt --lr_min --diag_shift`), and run it.** The network comes from `ARCH_FLAGS`
(the default `[4,4]/[4,4,1]` unless you overrode it in the configure cell). Each
writes `OUT_DIR/<NAME>.{json,mpack}` and logs to `WANDB_GROUP`; `--qgt dense` keeps
the SR solve exact and GPU-safe.

> Change `NAME` every run so outputs/W&B don't overwrite each other.

In [7]:
# === RUN one architecture (copy this cell per config; edit the arch lines) ===
NONINV = [8, 8]        # pre-Wilson blocks, UNIFORM channels: [4,4] = 2 blocks x 4 ch
INV    = [4, 4, 4]  # post-Wilson hidden widths AS DISPLAYED (trailing 1 = readout)

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ARCH_FLAGS = f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} " \
             f"--inv_hidden {' '.join(map(str, INV[:-1]))}"   # model appends final 1
NAME    = f"cnn-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR = f"outputs/{GROUP}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

arch cnn-8-8_4-4-4  |  --noninv_channels 8 --n_noninv 2 --inv_hidden 4 4  ->  outputs/colab-obc-hx0.2_hz0.2
/usr/local/lib/python3.12/dist-packages/netket/utils/struct/dataclass.py:394: FutureWarning: 
            You defined `__post_init__(self)` in a netket
            dataclass (a class decorated with @nk.utils.struct.dataclass) which
            inherits from a `nk.utils.struct.Pytree`.

            The class is <class 'simulation.custom_sampler.WeightedRule'>.

            This behaviour is not supported and might break. Please remove
            the decorator and just inherit from the base class, defining
            a standard `__init__` method which calls `super().__init__(...)`
            as usual.

            If you need help, reach out with us.
            
  warnings.warn(msg, category=FutureWarning, stacklevel=1)
NODE: b'72993b2600a1\n'
ASSIGNED DEVICE: b'NVIDIA L4\n'
NUMBER OF CHAINS: 1024
AVAILABLE DEVICES: [CudaDevice(id=0)]
/usr/local/lib/python3.12/dist-packages/net

In [8]:
# === RUN one architecture (copy this cell per config; edit the arch lines) ===
NONINV = [8, 8, 8]        # pre-Wilson blocks, UNIFORM channels: [4,4] = 2 blocks x 4 ch
INV    = [4, 4, 4]  # post-Wilson hidden widths AS DISPLAYED (trailing 1 = readout)

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ARCH_FLAGS = f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} " \
             f"--inv_hidden {' '.join(map(str, INV[:-1]))}"   # model appends final 1
NAME    = f"cnn-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR = f"outputs/{GROUP}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

arch cnn-8-8-8_4-4-4  |  --noninv_channels 8 --n_noninv 3 --inv_hidden 4 4  ->  outputs/colab-obc-hx0.2_hz0.2
/usr/local/lib/python3.12/dist-packages/netket/utils/struct/dataclass.py:394: FutureWarning: 
            You defined `__post_init__(self)` in a netket
            dataclass (a class decorated with @nk.utils.struct.dataclass) which
            inherits from a `nk.utils.struct.Pytree`.

            The class is <class 'simulation.custom_sampler.WeightedRule'>.

            This behaviour is not supported and might break. Please remove
            the decorator and just inherit from the base class, defining
            a standard `__init__` method which calls `super().__init__(...)`
            as usual.

            If you need help, reach out with us.
            
  warnings.warn(msg, category=FutureWarning, stacklevel=1)
NODE: b'72993b2600a1\n'
ASSIGNED DEVICE: b'NVIDIA L4\n'
NUMBER OF CHAINS: 1024
AVAILABLE DEVICES: [CudaDevice(id=0)]
/usr/local/lib/python3.12/dist-packages/n

In [9]:
# === RUN one architecture (copy this cell per config; edit the arch lines) ===
NONINV = [8]        # pre-Wilson blocks, UNIFORM channels: [4,4] = 2 blocks x 4 ch
INV    = [4, 4, 4]  # post-Wilson hidden widths AS DISPLAYED (trailing 1 = readout)

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ARCH_FLAGS = f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} " \
             f"--inv_hidden {' '.join(map(str, INV[:-1]))}"   # model appends final 1
NAME    = f"cnn-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR = f"outputs/{GROUP}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_full \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

arch cnn-8_4-4-4  |  --noninv_channels 8 --n_noninv 1 --inv_hidden 4 4  ->  outputs/colab-obc-hx0.2_hz0.2
/usr/local/lib/python3.12/dist-packages/netket/utils/struct/dataclass.py:394: FutureWarning: 
            You defined `__post_init__(self)` in a netket
            dataclass (a class decorated with @nk.utils.struct.dataclass) which
            inherits from a `nk.utils.struct.Pytree`.

            The class is <class 'simulation.custom_sampler.WeightedRule'>.

            This behaviour is not supported and might break. Please remove
            the decorator and just inherit from the base class, defining
            a standard `__init__` method which calls `super().__init__(...)`
            as usual.

            If you need help, reach out with us.
            
  warnings.warn(msg, category=FutureWarning, stacklevel=1)
NODE: b'72993b2600a1\n'
ASSIGNED DEVICE: b'NVIDIA L4\n'
NUMBER OF CHAINS: 1024
AVAILABLE DEVICES: [CudaDevice(id=0)]
/usr/local/lib/python3.12/dist-packages/netke

## CNN benchmark — geometry-exact convs, NO Wilson invariance

`ToricCNN_full`'s ~1e-5 errors need a baseline. `GeoCNN` uses the **same**
`KernelManager3D`/`GeoConv3D` kernel and the same depth, but **drops the Wilson
4-product** — so it is translation-equivariant but *not* `A_v`-invariant. A/B-ing
it against `ToricCNN_full` at matched params/depth isolates how much the *vertex
invariance* (not the geometry-exact gather) is worth. Works for OBC and PBC.

Run the param-count helper first to pick `CNN_HIDDEN` close to your full-net size,
then run the benchmark cell (copy it per config, change `CNN_HIDDEN`).

In [ ]:
# === Param-count helper: match GeoCNN to your ToricCNN_full ================
from Three_TC.builders import build_state

def _nparams(arch, **kw):
    cfg = {"L": L, "bc": BC, "hx": HX, "hz": HZ, "arch": arch,
           "n_samples": 256, "n_chains": 16, **kw}
    *_, vs, _ = build_state(cfg)
    return int(vs.n_parameters)

# your full-net reference (edit to match the ToricCNN_full you're comparing to)
print("ToricCNN_full [8,8]/[4,4,4]:",
      _nparams("ToricCNN_full", noninv_channels=8, n_noninv=2, inv_hidden=(4, 4)))
for h in [(8, 8, 8), (12, 12), (16, 8, 8)]:
    print(f"GeoCNN {h}:", _nparams("GeoCNN", cnn_hidden=h))

In [ ]:
# === RUN the CNN benchmark — geometry-exact convs, NO Wilson (copy per cfg) ==
CNN_HIDDEN = [8, 8, 8]   # edge-conv channel widths; depth = len+1 (width-1 readout)

# --- flags + tag (don't edit) ---------------------------------------------
cnn_flags = f"--cnn_hidden {' '.join(map(str, CNN_HIDDEN))}"
NAME    = f"geocnn-{'-'.join(map(str, CNN_HIDDEN))}"
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR = f"outputs/{GROUP}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {cnn_flags}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch GeoCNN \
  {FIELD_FLAGS} --n_iter {N_ITER} --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {cnn_flags} --out_dir {OUT_DIR} --name {NAME} {wb}

## Notes

**Vary hyperparameters:** copy a run cell, set `NAME`, and edit the training flags
(`--dt --lr_min --diag_shift`). To change the **network**, set `ARCH_FLAGS` once in
the configure cell (e.g. `--noninv_channels 8 --n_noninv 1 --inv_hidden 8`) and bump
`ARCH_TAG` so the new net's runs land in a fresh folder. Change the physical point
via `HZ_PRESET`.

**Why one cell per run:** each `!python` call is an isolated fresh process, which is
what runs reliably on Colab; the automated loop could skip runs, so it's unrolled.

**Outputs.** Each run writes `OUT_DIR/<NAME>.json` (config + curve + final
observables) and `OUT_DIR/<NAME>.mpack` (weights) — a fresh per-arch folder
(`outputs/colab-<preset>-<arch_tag>/`), so runs don't pile up with previous ones.
The Colab VM is ephemeral — download keepers with the last cell, or rely on W&B.

**Boundary conditions.** `BC = "OBC"` switches every run to open boundaries via `--bc OBC` and diagonalizes the L=2 OBC ground state on the fly for the `delta` FOM (no preset needed); PBC keeps using `--hz_preset`. `BC` is folded into `WANDB_GROUP`/`OUT_DIR`, so OBC and PBC runs never share a folder. The geometry-exact nets `ToricCNN`/`ToricCNN_full`/`GeoCNN` all support OBC (the Vanilla nets are PBC-only).

**CNN benchmark.** `GeoCNN` (the cells just above) is the no-Wilson control: same `GeoConv3D` kernel and depth as `ToricCNN_full` but without the `A_v`-invariant flux product. Match it to your full net with the param-count helper, then compare `delta` — a clear gap is the evidence that the Wilson invariance, not the kernel, drives the ~1e-5 errors.

In [ ]:
# Optional: zip + download THIS run folder before the VM recycles.
from google.colab import files
!zip -qr /content/{WANDB_GROUP}.zip {OUT_DIR} && echo "zipped {OUT_DIR}"
files.download(f"/content/{WANDB_GROUP}.zip")